In [2]:
%pip install pandas pyreadstat

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.3.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [4]:
import pandas as pd

# --- PHASE 1: Load Data (Bypassing pyreadstat) ---
print("Loading raw files...")
demo = pd.read_sas("data/DEMO_J.xpt", format="xport")
bio  = pd.read_sas("data/BIOPRO_J.xpt", format="xport")
cbc  = pd.read_sas("data/CBC_J.xpt", format="xport")
lux  = pd.read_sas("data/LUX_J.xpt", format="xport")
print("Files loaded successfully.\n")

# --- PHASE 2: Select & Merge ---
# 1. Isolate required markers
demo_s = demo[["SEQN", "RIDAGEYR"]]                    # Age
bio_s  = bio[["SEQN", "LBXSASSI", "LBXSATSI"]]         # AST, ALT
cbc_s  = cbc[["SEQN", "LBXPLTSI"]]                     # Platelets
lux_s  = lux[["SEQN", "LUXSMED"]]                      # Liver stiffness

# 2. Inner Join on Patient ID
merged = demo_s.merge(bio_s, on="SEQN").merge(cbc_s, on="SEQN").merge(lux_s, on="SEQN")

# 3. Save the master raw dataset
merged.to_csv("data/merged_raw.csv", index=False)

print("Merged shape:", merged.shape)
print("\nMissing values (NaNs) in the merged dataset:")
print(merged.isna().sum())

Loading raw files...
Files loaded successfully.

Merged shape: (6401, 6)

Missing values (NaNs) in the merged dataset:
SEQN          0
RIDAGEYR      0
LBXSASSI    519
LBXSATSI    499
LBXPLTSI    343
LUXSMED     451
dtype: int64


In [5]:
clean = merged.dropna(subset=["LUXSMED"]).copy()
print("Shape after dropping missing target:", clean.shape)
print(clean.isna().sum())

Shape after dropping missing target: (5950, 6)
SEQN          0
RIDAGEYR      0
LBXSASSI    421
LBXSATSI    402
LBXPLTSI    252
LUXSMED       0
dtype: int64


In [7]:
%pip install scikit-learn

   ---------------------------------------- 0.0/8.3 MB ? eta -:--:--
   --- ------------------------------------ 0.8/8.3 MB 4.8 MB/s eta 0:00:02
   ------ --------------------------------- 1.3/8.3 MB 3.4 MB/s eta 0:00:03
   ----------- ---------------------------- 2.4/8.3 MB 3.7 MB/s eta 0:00:02
   --------------- ------------------------ 3.1/8.3 MB 3.8 MB/s eta 0:00:02
   ------------------- -------------------- 3.9/8.3 MB 3.9 MB/s eta 0:00:02
   ---------------------- ----------------- 4.7/8.3 MB 3.9 MB/s eta 0:00:01
   --------------------------- ------------ 5.8/8.3 MB 4.0 MB/s eta 0:00:01
   ------------------------------ --------- 6.3/8.3 MB 4.0 MB/s eta 0:00:01
   -------------------------------- ------- 6.8/8.3 MB 3.8 MB/s eta 0:00:01
   ----------------------------------- ---- 7.3/8.3 MB 3.6 MB/s eta 0:00:01
   -------------------------------------- - 7.9/8.3 MB 3.4 MB/s eta 0:00:01
   ---------------------------------------  8.1/8.3 MB 3.3 MB/s eta 0:00:01
   ----------------


[notice] A new release of pip is available: 24.3.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [8]:
from sklearn.experimental import enable_iterative_imputer
from sklearn.impute import IterativeImputer

def impute_predictors(df):
    """MICE-impute AST, ALT, Platelets only — never the target."""
    cols_to_impute = ["LBXSASSI", "LBXSATSI", "LBXPLTSI"]
    imputer = IterativeImputer(random_state=42, max_iter=10)
    df[cols_to_impute] = imputer.fit_transform(df[cols_to_impute])
    print("Remaining NaNs after imputation:\n", df.isna().sum())
    return df

clean = impute_predictors(clean)

Remaining NaNs after imputation:
 SEQN        0
RIDAGEYR    0
LBXSASSI    0
LBXSATSI    0
LBXPLTSI    0
LUXSMED     0
dtype: int64


In [9]:
clean.to_csv("data/clean_imputed.csv", index=False)